# Recursive agents & budgets

_Compose nested jaz.invoke calls and cap cost across the whole tree._

⏱️ **~20 min** · Part of the [JAZ](https://jaz-lang.ai) tutorial series.

> Run the setup cells once, then work through the rest top to bottom.

## Setup

In [ ]:
# Install JAZ from PyPI
%pip install -q jaz-lang

In [ ]:
import os, getpass

# JAZ uses native OpenAI / Anthropic clients. Set at least one key.
# In Colab you can also use the secrets manager (key icon, left sidebar).
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')


In [ ]:
import jaz

# Pick the model all agents use. Format is 'provider/model'.
# Swap to an Anthropic model (and set ANTHROPIC_API_KEY) if you prefer.
jaz.configure(model_config={'model': 'openai/gpt-4o-mini'})

## Agents that call agents

An agent can call `jaz.invoke` from its own code, spawning nested agents. Use
`ConfigOverride` to allow recursion.

In [ ]:
data = ["the cat sat", "a dog ran", "birds fly south"]

result = jaz.invoke(
    """For each phrase, use jaz.invoke to count its words.
    Return the list of counts.""",
    jaz.ConfigOverride(max_repl_recursion=2),
    return_type=list,
    phrases=data,
)
result

## Capping cost across the tree

Pool budgets (total USD / call count) are enforced while `BudgetTrackingHook` is
active — they apply across every nested invoke.

In [ ]:
from jaz.hooks import BudgetTrackingHook

with BudgetTrackingHook(cost_budget=0.10, calls_budget=25):  # USD / calls
    result = jaz.invoke(
        "For each phrase, summarize it in one word using jaz.invoke.",
        jaz.ConfigOverride(max_repl_recursion=2),
        return_type=list,
        phrases=data,
    )
result

## Per-level limits

Recursion depth, invoke counts, and iterations per level are configuration:

In [ ]:
with jaz.config_override(
    max_repl_recursion=2,
    max_repl_invoke_calls=10,
    max_repl_iterations=20,
):
    result = jaz.invoke("A multi-step task", return_type=str)
result

Next: [observability with hooks](https://jaz-lang.ai/docs/tutorials).